In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['OPENAI_API_KEY']=os.getenv('OPENAI_API_KEY')
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')
os.environ['GOOGLE_API_KEY']=os.getenv('GOOGLE_API_KEY')

In [2]:


from langchain_community.document_loaders import PyPDFDirectoryLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_google_genai import chat_models,ChatGoogleGenerativeAI

/var/folders/7g/w4x0jgw150nghjpzv68w5sdh0000gn/T/ipykernel_41559/3015656914.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader,PyPDFLoader
/Users/adityakumar/Developer/Code/genai/langchain_projects/text_summarization/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:

loader= PyPDFLoader('source_docs/transformers-attention.pdf')
docs=loader.load()

splitter=RecursiveCharacterTextSplitter(chunk_size=2000,chunk_overlap=200)
splited=splitter.split_documents(docs)


In [4]:
len(splited)

27

### Do not execute below this

In [5]:
import o

ModuleNotFoundError: No module named 'o'

In [ ]:




chunk_summary_generic_prompt = """
You are an expert information extraction system.

Your job is NOT to summarize.

Your job is to convert the document chunk into a structured information record.

Extract and preserve:

- headings
- subheadings
- definitions
- concepts
- facts
- claims
- methodologies
- algorithms
- formulas
- equations
- numerical values
- hyperparameters
- experimental settings
- datasets
- tables
- examples
- conclusions
- limitations
- future work

Rules:

- Preserve information verbatim whenever possible.
- Do not explain.
- Do not simplify.
- Do not compress numerical information.
- Do not merge concepts.
- Do not omit technical details.
- Remove only exact repetition.

Output structured markdown.
"""

chunk_summary_prompt=ChatPromptTemplate.from_messages(
    [
        ('system',chunk_summary_generic_prompt),
        
        ('human','{text}')
    ]
)

#llm=ChatOpenAI(model='gpt-4o-mini',temperature=0.0)
#llm=ChatGroq(model='llama-3.1-8b-instant')
llm=ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite')
chunk_summ_chain=chunk_summary_prompt|llm

import time
all_chunks = []

for i, chunk in enumerate(splited, start=1):
    try:
        result = chunk_summ_chain.invoke(
            {"text": chunk.page_content}
        )

        all_chunks.append(result.content[0]['text'])
        print(result.content)

        print(f"Chunk {i} done")

    except Exception as e:
        print(f"Chunk {i} failed:", e)
        time.sleep(20)


reducer_prompt = """
You are an expert document consolidation system.

You are given structured information extracted from document chunks.

Your task:

1. Merge overlapping information.
2. Remove duplicate statements.
3. Preserve every unique fact.
4. Preserve every formula.
5. Preserve every numerical value.
6. Preserve every experiment result.
7. Preserve every table entry.
8. Preserve every definition.
9. Preserve all section hierarchy.

Rules:

- Never remove information because it seems unimportant.
- Never generalize specific values.
- Never replace numbers with descriptions.
- Never drop formulas.
- Never drop hyperparameters.
- Never drop examples.
- Never invent missing information.

Output:

# Detailed Summary

Organize the information using markdown headings and subheadings.
"""

summarizer_prompt=ChatPromptTemplate.from_messages([
    ('system',reducer_prompt) ,
    ('human','document : {text}')
      
        ])

reducer_chain=summarizer_prompt|llm

chunked_sum='\n'.join([chunk for chunk in all_chunks])

response=reducer_chain.invoke({'text':chunked_sum,'language':'hindi'})
print(response.content[0]['text'])

In [ ]:
chunk_summary_generic_prompt = """
You are an expert information extraction system.

Your job is NOT to summarize.

Your job is to convert the document chunk into a structured information record.

Extract and preserve:

- headings
- subheadings
- definitions
- concepts
- facts
- claims
- methodologies
- algorithms
- formulas
- equations
- numerical values
- hyperparameters
- experimental settings
- datasets
- tables
- examples
- conclusions
- limitations
- future work

Rules:

- Preserve information verbatim whenever possible.
- Do not explain.
- Do not simplify.
- Do not compress numerical information.
- Do not merge concepts.
- Do not omit technical details.
- Remove only exact repetition.

Output structured markdown.
"""

chunk_summary_prompt=ChatPromptTemplate.from_messages(
    [
        ('system',chunk_summary_generic_prompt),
        
        ('human','{text}')
    ]
)

In [ ]:
reducer_prompt = """
You are an expert document consolidation system.

You are given structured information extracted from document chunks.

Your task:

1. Merge overlapping information.
2. Remove duplicate statements.
3. Preserve every unique fact.
4. Preserve every formula.
5. Preserve every numerical value.
6. Preserve every experiment result.
7. Preserve every table entry.
8. Preserve every definition.
9. Preserve all section hierarchy.

Rules:

- Never remove information because it seems unimportant.
- Never generalize specific values.
- Never replace numbers with descriptions.
- Never drop formulas.
- Never drop hyperparameters.
- Never drop examples.
- Never invent missing information.

Output:

# Detailed Summary

Organize the information using markdown headings and subheadings.
"""

summarizer_prompt=ChatPromptTemplate.from_messages([
    ('system',reducer_prompt) ,
    ('human','document : {text}')
      
        ])